In [10]:
import numpy as np
import pymc as pm
import arviz as az

# Generate synthetic daily data with trend + weekly seasonality
rng = np.random.default_rng(42)
n_days = 365
t = np.arange(n_days, dtype=float)

trend = 0.05 * t
seasonality = 3 * np.sin(2 * np.pi * t / 7)
noise = rng.normal(0, 1, n_days)
y = 10 + trend + seasonality + noise

with pm.Model() as forecast_model:
    # Priors
    intercept = pm.Normal("intercept", mu=0, sigma=10)
    slope = pm.Normal("slope", mu=0, sigma=1)
    amp = pm.HalfNormal("amplitude", sigma=5)
    period = pm.Normal("period", mu=7, sigma=0.5)
    sigma = pm.HalfNormal("sigma", sigma=2)

    # Deterministic trend + seasonality
    mu = intercept + slope * t + amp * pm.math.sin(2 * np.pi * t / period)

    # Likelihood
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

    # Sample
    trace = pm.sample(2000, tune=1000, cores=2, random_seed=42)

az.summary(trace, var_names=["intercept", "slope", "amplitude", "period", "sigma"])

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, slope, amplitude, period, sigma]


Output()

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
intercept,9.923,0.101,9.721,10.099,0.002,0.002,2245.0,2798.0,1.0
slope,0.050,0.000,0.049,0.051,0.000,0.000,2331.0,2584.0,1.0
amplitude,3.086,0.070,2.950,3.212,0.001,0.001,3916.0,2822.0,1.0
period,6.999,0.001,6.997,7.000,0.000,0.000,4343.0,3259.0,1.0
sigma,0.944,0.035,0.877,1.010,0.001,0.001,3876.0,2678.0,1.0


In [11]:
import pickle
trace.to_netcdf("model_artifacts/trace.nc")

'model_artifacts/trace.nc'

In [12]:
artifacts = {
    "n_train": n_days,
    "t_train": t,
    "y_train": y,
}

with open("model_artifacts/metadata.pkl", "wb") as f:
    pickle.dump(artifacts, f)